In [ ]:
# Monte Carlo synthesis of the Milky Way stellar population
#
# Pipeline: initial masses from a Kroupa IMF (rejection sampling or inverse transform),
# ages from a constant star formation rate, birth metallicities from a closed-box model,
# and remnants from initial-final mass relations. Each simulation is saved to
# remnants_{method}_N{N}.dat, so all the analysis cells only read files.
#
# Run order: first the model and simulation cells, then the analysis cells. Simulations
# already saved with the same parameters are skipped, so re-running the notebook is fast.

import numpy as np
import matplotlib.pyplot as plt
import random
import os
import glob
import re
import time
plt.rcdefaults()

# np.trapz is called np.trapezoid in NumPy >= 2.0
integrate = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz

In [ ]:
#Parameters

# Reproducibility
SEED = 12345          # base random seed
REDO = False          # True: simulate again even if the file already exists

# Population sizes
n = [10, 100, 1000, 10000, 100000, 1000000]

# IMF sampling method: 'rejection' or 'inverse' (inverse transform)
METHOD = 'rejection'

# IMF [Msun]
M_min = 0.08
M_max = 120

# Galaxy and chemical evolution
T_MW = 10                  # age of the MW [Gyr]
u_f = 0.1                  # final gas fraction mu_f
Z_sun = 0.014              # solar metallicity
y = Z_sun/np.log(1/u_f)    # effective yield, so that Z_gas(T_MW) = Z_sun

# Remnants
M_up             = 9.0    # WD -> NS transition [Msun]
f_ej             = 0.9    # ejection fraction (Raithel+18, Eq. 4)
p_BH_15_217      = 0.4    # BH probability in 15-21.7 Msun (Raithel+18, Fig. 2)
p_BH_252_275     = 0.25   # BH probability in 25.2-27.5 Msun (Raithel+18, Fig. 2)
p_BH_60_120      = 0.6    # BH probability in 60-120 Msun (Raithel+18, Fig. 2)
NS_gravitational = True   # baryonic -> gravitational mass (Raithel+18, Eqs. 19-20)
R_NS             = 12.0   # NS radius [km]

# MDF
M_G_min, M_G_max = 0.8, 1.05            # G-dwarf mass range [Msun]
bins_FeH = np.arange(-2.5, 0.7, 0.1)    # same bins for model and observed MDFs
metal_poor_threshold = -0.5             # [Fe/H] below which a star is metal-poor

# Infall
tau_infall_list = [2, 5, 8]   # accretion timescales [Gyr]
M_0_infall = 0.1              # initial gas mass, in units of M_tot(T_MW)
Z_in = 0.0                    # metallicity of the accreted gas
recalibrate_yield = True      # True: y is adjusted so that Z_gas(T_MW) = Z_sun
n_t_chem = 20001              # time steps of the chemical evolution integration


def simulation_seed(N, method):
    """
    Random seed of the simulation (N, method).

    Each combination gets its own seed, so every simulation is reproducible on its own,
    regardless of which other simulations are run. The offset keeps the two methods independent.
    """
    return SEED + N + (0 if method == 'rejection' else 10_000_000)


def parameter_signature(method):
    """
    Text with all the parameters that affect a simulation.

    It is written as the first header line of each file. A file is reused only if this
    line matches the current parameters (see file_up_to_date).
    """
    return (f'SEED={SEED} sampling={method} T_MW={T_MW} M_min={M_min} M_max={M_max} '
            f'u_f={u_f} Z_sun={Z_sun} M_up={M_up} f_ej={f_ej} '
            f'p_BH=({p_BH_15_217},{p_BH_252_275},{p_BH_60_120}) NS_grav={NS_gravitational} R_NS={R_NS}')

In [ ]:
def IMF(m):
    """
    Kroupa (2001) IMF, xi(m), not normalized.

    Piecewise power law with slopes 0.3, 1.3 and 2.3. The constant C = 0.5 makes xi(m)
    continuous at 0.5 Msun. The segments 0.5-1 and >= 1 Msun share the same slope and constant.

    Input:  m, mass [Msun]
    Output: xi(m)
    """
    C = 0.5**(-1.3)/0.5**(-2.3)
    if 0.01 <= m < 0.08:
        a = 0.3
        IMF = m**(-a)
        return IMF
    elif 0.08 <= m < 0.50:
        a = 1.3
        IMF = m**(-a)
        return IMF
    elif 0.50 <= m < 1.0:
        a = 2.3
        IMF = C*m**(-a)
        return IMF
    elif 1 <= m:
        a = 2.3
        IMF = C*m**(-a)
        return IMF
    else:
        return print('Invalid mass range')


# Inverse transform sampling
# In [M_min, M_max] the IMF has two segments xi(m) = k m^(-alpha). The probability of each
# segment is its relative area, P_i = I_i / (I_1 + I_2). A uniform number U first selects the
# segment and is then rescaled to [0, 1] to invert the cumulative distribution inside it.

# IMF segments: lower limit, upper limit, constant, slope
a1, b1, k1, alpha1 = M_min, 0.5,   1.0, 1.3
a2, b2, k2, alpha2 = 0.5,   M_max, 0.5, 2.3

# Integral of each segment
I1 = k1 * (b1**(1-alpha1) - a1**(1-alpha1)) / (1-alpha1)
I2 = k2 * (b2**(1-alpha2) - a2**(1-alpha2)) / (1-alpha2)

# Probability of each segment
P1 = I1 / (I1 + I2)
P2 = I2 / (I1 + I2)


def inverse_transform_mass(U):
    """
    Initial mass from the inverse of the cumulative IMF, M = F^-1(U).

    Input:  U, uniform random number in [0, 1)
    Output: initial mass [Msun]
    """
    if U < P1:
        u = U / P1
        a, b, alpha = a1, b1, alpha1
    else:
        u = (U - P1) / P2
        a, b, alpha = a2, b2, alpha2
    return (a**(1-alpha) + u*(b**(1-alpha) - a**(1-alpha)))**(1/(1-alpha))

In [ ]:
#Population simulation
def simulate_population(N, method):
    """
    Generates a population of N stars with initial mass, age and birth metallicity.

    Masses:
      'rejection': a mass is proposed uniformly in [M_min, M_max] and accepted if a second
                   uniform number is below xi(M)/xi(M_min). Since the IMF decreases with mass,
                   xi(M_min) is its maximum, so the ratio is between 0 and 1.
      'inverse':   inverse transform, one random number per star (see inverse_transform_mass).
    Ages: uniform in [0, T_MW] (constant star formation rate), with t_birth = T_MW - t_age.
    Metallicity: closed box, Z_gas = y ln(1/mu), with mu decreasing linearly from 1 to u_f.

    Inputs:  N, number of stars; method, 'rejection' or 'inverse'
    Outputs: lists of masses [Msun], ages [Gyr], birth times [Gyr] and [M/H],
             and the total number of iterations (mass proposals; equal to N for 'inverse')
    """
    masses = []
    ages = []
    birth_times = []
    metallicities = []
    count = 0
    count_total = 0

    while count < N:
        if method == 'rejection':
            random_mass = random.uniform(M_min, M_max)
            IMF_norm = IMF(random_mass)/IMF(M_min)
            IMF_random = random.uniform(0, 1)
            if IMF_random > IMF_norm:
                count_total = count_total + 1
                continue
        else:
            random_mass = inverse_transform_mass(random.uniform(0, 1))

        masses.append(random_mass)
        t_age = random.uniform(0,T_MW) #t_age
        t_birth = T_MW - t_age # t_birth
        ages.append(t_age)
        birth_times.append(t_birth)
        u = 1 - (1 - u_f) * t_birth / T_MW
        Z_gas = y * np.log(1/u)
        Z_birth = Z_gas
        M_H = np.log10(Z_birth/Z_sun)
        metallicities.append(M_H)
        count = count + 1

    return masses, ages, birth_times, metallicities, count + count_total

In [ ]:
# Remnants from initial-final mass relations
# type: 0 = still on the MS, 1 = WD, 2 = NS, 3 = BH, -1 = no relation for that mass

def t_MS(M):
    """Main-sequence lifetime [Gyr] of a star of initial mass M [Msun]."""
    return 10 * M**(-2.5)


# White dwarfs: Miller et al. (2026), Full Sample Fit 1, Eqs. 6-9 
def WD_mass(M):
    """
    White dwarf mass [Msun] from the initial mass M [Msun].

    The fit is valid for 0.84-8.39 Msun; Eq. 9 is extrapolated up to M_up to close the gap
    with the neutron star relations, which start at 9 Msun.
    """
    if 0.84 <= M < 1.98:
        return 0.160*M + 0.394      # Eq. 6
    elif 1.98 <= M < 2.33:
        return -0.155*M + 1.015     # Eq. 7
    elif 2.33 <= M < 3.98:
        return 0.174*M + 0.251      # Eq. 8
    elif 3.98 <= M < M_up:
        return 0.075*M + 0.644      # Eq. 9 (extrapolated above 8.39 if M_up > 8.39)
    else:
        return np.nan


# Neutron stars: Raithel, Sukhbold & Özel (2018), Sec. 5 
def NS_baryonic_mass(M):
    """
    Neutron star baryonic mass [Msun] from the initial mass M [Msun].

    Branches V and VII are Gaussian distributions, so a random value is drawn.
    """
    if M_up <= M <= 13:
        x = M - 14.75
        return 2.24 + 0.508*x + 0.125*x**2 + 0.0110*x**3          # Eq. 10 (branch I)
    elif 13 < M < 15:
        return 0.123 + 0.112*M                                    # Eq. 11 (branch II)
    elif 15 <= M < 17.8:
        return 0.996 + 0.0384*M                                   # Eq. 12 (branch III)
    elif 17.8 <= M < 18.5:
        return -0.020 + 0.10*M                                    # Eq. 13 (branch IV)
    elif 18.5 <= M < 21.7:
        return random.gauss(1.60, 0.155)                          # Eq. 14 (branch V, Gaussian)
    elif 25.2 <= M < 27.5:
        x = M - 2.619
        return 3232.29 - 409.429*x + 17.2867*x**2 - 0.24315*x**3  # Eq. 15 (branch VI)
    elif 60 <= M <= 120:
        return random.gauss(1.78, 0.016)                          # branch VII (Gaussian)
    else:
        return np.nan


def NS_gravitational_mass(M_b):
    """
    Gravitational mass [Msun] of a neutron star with baryonic mass M_b [Msun].

    Solves M_b = M_g + BE, with the binding energy BE = M_g 0.6 beta / (1 - 0.5 beta) and
    beta = G M_g / (R c^2) (Raithel+18, Eqs. 19-20). Since M_g appears on both sides, it is
    found by fixed-point iteration starting from M_g = M_b.
    """
    G = 6.674e-8          # cgs
    c = 2.998e10          # cm/s
    M_sol = 1.989e33      # g
    R = R_NS * 1e5        # cm
    M_g = M_b
    for _ in range(50):   # fixed-point iteration
        beta = G * M_g * M_sol / (R * c**2)
        M_g = M_b / (1 + 0.6*beta / (1 - 0.5*beta))
    return M_g


def NS_mass(M):
    """Neutron star mass [Msun]: gravitational if NS_gravitational is True, baryonic otherwise."""
    M_b = NS_baryonic_mass(M)
    if NS_gravitational:
        return NS_gravitational_mass(M_b)
    return M_b


# Black holes: Raithel, Sukhbold & Özel (2018), Sec. 4 
def BH_mass(M):
    """
    Black hole mass [Msun] from the initial mass M [Msun].

    15-40 Msun (red supergiants): Eq. 4 mixes the He-core collapse (Eq. 1) and the full collapse
    (Eq. 2) with the ejection fraction f_ej. 45-120 Msun (Wolf-Rayet stars, envelope already
    lost): CO-core collapse, Eq. 3. The 40-45 Msun gap is linearly interpolated.
    """
    if 15 <= M <= 40:
        # Branch I (red supergiants)
        M_core = -2.024 + 0.4130*M                                          # Eq. 1 (He core)
        x = M - 21.872
        M_all = 16.28 + 0.00694*x - 0.05973*x**2 + 0.003112*x**3            # Eq. 2 (full collapse)
        return f_ej*M_core + (1 - f_ej)*M_all                               # Eq. 4
    elif 40 < M < 45:
        # Gap between branches I and II: linear interpolation
        M_40 = BH_mass(40)
        M_45 = BH_mass(45)
        return M_40 + (M_45 - M_40) * (M - 40) / 5
    elif 45 <= M <= 120:
        # Branch II (Wolf-Rayet, no H envelope)
        return 5.795 + 1.007e9 * M**(-4.926)                                # Eq. 3
    else:
        return np.nan


# Remnant assignment
def remnant(M, t_age):
    """
    Type and present-day mass of a star of initial mass M [Msun] and age t_age [Gyr].

    If t_age < t_MS the star is still on the main sequence and keeps its initial mass.
    Otherwise the remnant depends on M. In the mixed ranges a uniform random number is
    compared with the BH probability, p_BH = 1 - f_NS (Raithel+18, Fig. 2).

    Output: (type, mass [Msun])
    """
    if t_age < t_MS(M):
        return 0, M                                   # still on the main sequence
    if M < M_up:
        return 1, WD_mass(M)                          # WD
    if M < 15:
        return 2, NS_mass(M)                          # NS
    if 15 <= M < 21.7:                                # mixed NS / BH
        return (3, BH_mass(M)) if random.random() < p_BH_15_217 else (2, NS_mass(M))
    if 21.7 <= M < 25.2:
        return 3, BH_mass(M)                          # BH
    if 25.2 <= M < 27.5:                              # mixed NS / BH
        return (3, BH_mass(M)) if random.random() < p_BH_252_275 else (2, NS_mass(M))
    if 27.5 <= M < 60:
        return 3, BH_mass(M)                          # BH
    if 60 <= M <= 120:                                # mixed NS / BH
        return (3, BH_mass(M)) if random.random() < p_BH_60_120 else (2, NS_mass(M))
    return -1, np.nan                                 # outside the range of the relations (M > 120)

In [ ]:
# Simulation loop, run for different population sizes

def file_up_to_date(N, method):
    """
    True if remnants_{method}_N{N}.dat exists and was generated with the current parameters.

    It compares the first header line of the file with parameter_signature(method).
    """
    filename = f'remnants_{method}_N{N}.dat'
    if not os.path.exists(filename):
        return False
    with open(filename, 'r') as f:
        return f.readline().strip() == '# ' + parameter_signature(method)


def population_synthesis(method):
    """
    Simulates and saves the populations of all the sizes in n with one sampling method.

    For each N: if the file already exists with the same parameters it is skipped (unless
    REDO = True); otherwise the seed is set, the population is generated, a remnant is assigned
    to each star and the result is saved to remnants_{method}_N{N}.dat.

    File header: parameter signature, iterations and runtime, column names and type codes.
    Columns: initial mass [Msun], age [Gyr], [M/H], type, present-day mass [Msun].
    """
    for N in n:
        if file_up_to_date(N, method) and not REDO:
            print(f'{method}, N = {N}: already exists, skipped')
            continue

        random.seed(simulation_seed(N, method))
        t0 = time.perf_counter()
        masses, ages, birth_times, metallicities, iterations = simulate_population(N, method)

        types = []
        remnant_masses = []
        for M, t in zip(masses, ages):
            obj_type, M_rem = remnant(M, t)
            types.append(obj_type)
            remnant_masses.append(M_rem)
        runtime = time.perf_counter() - t0

        np.savetxt(f'remnants_{method}_N{N}.dat',
                   np.column_stack([masses, ages, metallicities, types, remnant_masses]),
                   fmt=['%.6f', '%.6f', '%.6f', '%d', '%.6f'],
                   header=parameter_signature(method) + '\n'
                          f'iterations = {iterations} runtime_s = {runtime:.2f}\n'
                          'M[Msun] age[Gyr] [M/H] type M_rem[Msun]\n'
                          'type: 0=MS, 1=WD, 2=NS, 3=BH, -1=no relation')

        types = np.array(types)
        print(f'{method}, N = {N}: MS = {np.sum(types == 0)}, WD = {np.sum(types == 1)}, '
              f'NS = {np.sum(types == 2)}, BH = {np.sum(types == 3)}, no relation = {np.sum(types == -1)}, '
              f'iterations = {iterations}, runtime = {runtime:.1f} s')

population_synthesis(METHOD)

In [ ]:
# Simulations with both sampling methods, used in the method comparison
population_synthesis('rejection')
population_synthesis('inverse')

In [ ]:
# Chemical evolution with infall: A(t) = A0 exp(-t/tau)

def chemical_evolution(tau, M_0, Z_in=Z_in, recalibrate=recalibrate_yield):
    """
    Gas metallicity Z_gas(t) of a one-zone model with exponential infall.

    Masses are in units of M_tot(T_MW). The star formation rate is constant, as in the base
    model, so gas is locked into stars at psi = (1 - u_f) / T_MW. Conservation of gas and metals,
        dM_gas/dt = -psi + A(t),
        d(Z M_gas)/dt = y psi - Z psi + Z_in A(t),
    gives dZ/dt = [y psi - (Z - Z_in) A(t)] / M_gas(t), which has no closed-form solution with
    exponential infall and is integrated with a fourth-order Runge-Kutta method.
    A0 is fixed so that the accreted mass is 1 - M_0, which gives mu(T_MW) = u_f.

    Inputs:  tau, accretion timescale [Gyr] (None = closed box, use M_0 = 1);
             M_0, initial gas mass; Z_in, metallicity of the accreted gas;
             recalibrate, True to adjust y so that Z_gas(T_MW) = Z_sun
    Output:  dictionary with t [Gyr], Z(t), M_gas(t), M_tot(t), A(t) and the yield y used
    """
    psi = (1 - u_f) / T_MW
    A0 = 0.0 if tau is None else (1 - M_0) / (tau * (1 - np.exp(-T_MW / tau)))

    # Accretion rate
    def A(t):
        return 0.0 * t if tau is None else A0 * np.exp(-t / tau)

    # Accreted mass
    def M_acc(t):
        return 0.0 * t if tau is None else A0 * tau * (1 - np.exp(-t / tau))

    # Gas mass
    def M_gas(t):
        return M_0 + M_acc(t) - psi * t

    t = np.linspace(0, T_MW, n_t_chem)
    if np.any(M_gas(t) <= 0):
        raise ValueError(f'tau = {tau}, M_0 = {M_0}: the gas mass becomes negative')

    # Runge-Kutta 4 for dZ/dt = [y psi - (Z - Z_in) A] / M_gas, starting from Z(0) = 0
    def integrate_Z(y_val):
        f = lambda s, Z: (y_val * psi - (Z - Z_in) * A(s)) / M_gas(s)
        dt = t[1] - t[0]
        Z = np.zeros_like(t)
        for i in range(len(t) - 1):
            s, z = t[i], Z[i]
            k1 = f(s, z)
            k2 = f(s + dt/2, z + dt/2 * k1)
            k3 = f(s + dt/2, z + dt/2 * k2)
            k4 = f(s + dt, z + dt * k3)
            Z[i+1] = z + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
        return Z

    # Yield adjusted so that Z_gas(T_MW) = Z_sun. Z(t) is linear in y, Z = Z_0 + y (Z_1 - Z_0),
    # so two integrations (y = 0 and y = 1) are enough to find it
    if recalibrate:
        Z_0 = integrate_Z(0.0)
        Z_1 = integrate_Z(1.0)
        y_mod = (Z_sun - Z_0[-1]) / (Z_1[-1] - Z_0[-1])
        Z = Z_0 + y_mod * (Z_1 - Z_0)
    else:
        y_mod = y
        Z = integrate_Z(y)

    return {'t': t, 'Z': Z, 'M_gas': M_gas(t), 'M_tot': M_0 + M_acc(t), 'A': A(t), 'y': y_mod}


def birth_MH(model, age):
    """
    Birth [M/H] of stars with age t_age [Gyr] in a chemical evolution model.

    Interpolates Z_gas at t_birth = T_MW - t_age. Masses, ages and remnants do not depend on the
    chemistry, so the same simulated stars are reused for every model.
    """
    t_birth = T_MW - age
    with np.errstate(divide='ignore'):
        return np.log10(np.interp(t_birth, model['t'], model['Z']) / Z_sun)


# Closed box (no infall) and one model per tau
models = {'Closed box': chemical_evolution(None, 1.0)}
for tau in tau_infall_list:
    models[rf'Infall, $\tau$ = {tau} Gyr'] = chemical_evolution(tau, M_0_infall)

# Validation: the integrated closed box must match the analytic solution Z = y ln(1/mu)
cb = models['Closed box']
Z_analytic = y * np.log(cb['M_tot'][:-1] / cb['M_gas'][:-1])
print(f'Closed-box validation: maximum relative difference = {np.max(np.abs(cb["Z"][:-1] - Z_analytic) / Z_analytic[-1]):.2e}')
for name, mod in models.items():
    print(f'{name}: y = {mod["y"]:.5f}, mu(T_MW) = {mod["M_gas"][-1]/mod["M_tot"][-1]:.3f}, '
          f'Z_gas(T_MW)/Z_sun = {mod["Z"][-1]/Z_sun:.3f}')

### Analysis and figures

From here on, the cells only read the saved simulation files (`remnants_{method}_N{N}.dat`) and the observed MDFs (`MDF_obs_*.dat`) to compute the tables and generate the figures; no new simulations are run. This section was written with the help of Claude Code for efficiency. The previous section was optimized with Claude Cowork, but the structure and logic of the infall model and of the remnant assignment are the authors' own.

In [ ]:
# Reading the files of each simulation
type_names = {0: 'MS', 1: 'WD', 2: 'NS', 3: 'BH'}
type_colors = {0: 'goldenrod', 1: 'royalblue', 2: 'darkorange', 3: 'black'}


def list_N(method=METHOD):
    """Sorted list of the N that have a file remnants_{method}_N{N}.dat in the folder."""
    files = glob.glob(f'remnants_{method}_N*.dat')
    return sorted(int(re.search(r'_N(\d+)\.dat', a).group(1)) for a in files)


def load_simulation(N, method=METHOD):
    """
    Reads remnants_{method}_N{N}.dat.

    Output: initial mass [Msun], age [Gyr], [M/H], type, present-day mass [Msun]
            (arrays) and the number of iterations stored in the header.
    """
    filename = f'remnants_{method}_N{N}.dat'
    data = np.loadtxt(filename, ndmin=2)
    with open(filename, 'r') as f:
        f.readline()
        iterations = int(f.readline().split()[3])
    # columns: M, age, [M/H], type, M_rem
    return data[:, 0], data[:, 1], data[:, 2], data[:, 3], data[:, 4], iterations


def read_runtime(N, method):
    """Runtime [s] of the simulation stored in the header of remnants_{method}_N{N}.dat."""
    with open(f'remnants_{method}_N{N}.dat', 'r') as f:
        f.readline()
        return float(f.readline().split()[6])


N_list = list_N()
print('N found:', N_list)

In [ ]:
# IMF validation as a function of N
# The histograms use logarithmic bins and density=True, which divides each count by the bin
# width, so they give dN/dM and can be compared directly with the IMF normalized to unit area.

m_grid = np.logspace(np.log10(M_min), np.log10(M_max), 2000)
m_grid[0], m_grid[-1] = M_min, M_max      # avoids rounding the first point below M_min
IMF_grid = np.array([IMF(m) for m in m_grid])
IMF_grid = IMF_grid / integrate(IMF_grid, m_grid)

bins = np.logspace(np.log10(M_min), np.log10(M_max), 50)

# Mosaic with one panel per N
ncols = 3
nrows = int(np.ceil(len(N_list) / ncols))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(N_list)))

fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).flatten()

for i, (N, color) in enumerate(zip(N_list, colors)):
    ax = axes[i]
    M, age, MH, obj_type, M_rem, iterations = load_simulation(N)

    ax.hist(M, bins=bins, density=True, alpha=0.7, color=color, edgecolor='white', label=f'N = {N}')
    ax.plot(m_grid, IMF_grid, color='black', label='Kroupa IMF')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.legend(fontsize=14, loc='upper right')
    ax.tick_params(labelsize=14, labelbottom=True)
    if i % ncols == 0:
        ax.set_ylabel(r'$dN/dM$ (normalized)', fontsize=18)
    if i >= len(N_list) - ncols:
        ax.set_xlabel(r'Mass [$M_\odot$]', fontsize=18)

# Removes the empty panels when len(N_list) is not a multiple of ncols
for j in range(len(N_list), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig("IMF_vs_N.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Convergence: number and fraction of MS, WD, NS and BH as a function of N
counts = {k: [] for k in type_names}
fractions = {k: [] for k in type_names}

for N in N_list:
    M, age, MH, obj_type, M_rem, iterations = load_simulation(N)
    for k in type_names:
        counts[k].append(np.sum(obj_type == k))
        fractions[k].append(np.sum(obj_type == k) / len(obj_type))
    print(f'N = {N}: ' + ', '.join(f'{type_names[k]} = {fractions[k][-1]:.5f}' for k in type_names))

# Grouped bars: one group per N and one bar per type
x = np.arange(len(N_list))
width = 0.2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for i, k in enumerate(type_names):
    ax1.bar(x + (i - 1.5)*width, counts[k], width, color=type_colors[k], label=type_names[k])
    ax2.bar(x + (i - 1.5)*width, fractions[k], width, color=type_colors[k], label=type_names[k])

# Number per stellar type
ax1.set_yscale('log')
ax1.set_xticks(x)
ax1.set_xticklabels([f'{N}' for N in N_list], fontsize=14)
ax1.set_xlabel('N', fontsize=18)
ax1.set_ylabel('Number of objects', fontsize=18)
ax1.legend(fontsize=14)

# Fraction per stellar type
ax2.set_yscale('log')
ax2.set_xticks(x)
ax2.set_xticklabels([f'{N}' for N in N_list], fontsize=14)
ax2.set_xlabel('N', fontsize=18)
ax2.set_ylabel('Fraction', fontsize=18)
ax2.legend(fontsize=14)

plt.tight_layout()
plt.savefig("CONVERGENCE.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Initial and final mass distributions per type (largest N)
# For MS stars the final mass is the initial mass, since they have not evolved yet.
N = N_list[-1]
M, age, MH, obj_type, M_rem, iterations = load_simulation(N)

bins = np.logspace(np.log10(M_min), np.log10(M_max), 50)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for k in type_names:
    sel = obj_type == k
    ax1.hist(M[sel], bins=bins, histtype='step', linewidth=2, color=type_colors[k],
             label=f'{type_names[k]} ({np.sum(sel)})')
    ax2.hist(M_rem[sel], bins=bins, histtype='step', linewidth=2, color=type_colors[k],
             label=f'{type_names[k]} ({np.sum(sel)})')

for ax in (ax1, ax2):
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.legend(fontsize=14)
    ax.tick_params(labelsize=14)

ax1.set_xlabel(r'Initial mass [$M_\odot$]', fontsize=18)
ax2.set_xlabel(r'Final mass [$M_\odot$]', fontsize=18)
ax1.set_ylabel('Number of objects', fontsize=18)

plt.tight_layout()
plt.savefig("MASSES.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Age distribution of the present-day population per type (largest N)
# density=True normalizes each type to unit area, so the shapes can be compared even though
# the number of objects of each type differs by orders of magnitude.
N = N_list[-1]
M, age, MH, obj_type, M_rem, iterations = load_simulation(N)

bins = np.linspace(0, T_MW, 41)

plt.figure(figsize=(10, 6))
for k in type_names:
    sel = obj_type == k
    plt.hist(age[sel], bins=bins, histtype='step', linewidth=2, density=True,
             color=type_colors[k], label=f'{type_names[k]} ({np.sum(sel)})')

plt.xlabel('Age [Gyr]', fontsize=12)
plt.ylabel('Normalized distribution', fontsize=12)
plt.legend(fontsize=10,loc='lower right')

plt.savefig("AGES.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Age-metallicity relation + closed-box input relation
# The red curve is the closed-box relation used to assign the metallicities, so the simulated
# stars must lie on it; only the sampling of the curve changes with N.
t_curve = np.linspace(0, T_MW, 500)[:-1]        # t_age = T_MW is excluded, where Z = 0
u_curve = 1 - (1 - u_f) * (T_MW - t_curve) / T_MW
MH_curve = np.log10(y * np.log(1/u_curve) / Z_sun)

# Mosaic with one panel per N
ncols = 3
nrows = int(np.ceil(len(N_list) / ncols))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(N_list)))

fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).flatten()

for i, (N, color) in enumerate(zip(N_list, colors)):
    ax = axes[i]
    M, age, MH, obj_type, M_rem, iterations = load_simulation(N)

    ax.plot(age, MH, '.', color=color, alpha=0.7,
            label=f'N = {N}\niterations = {iterations}')
    ax.plot(t_curve, MH_curve, color='red', label='Closed box')
    ax.legend(fontsize=14, loc='lower left')
    ax.tick_params(labelsize=14, labelbottom=True)
    if i % ncols == 0:
        ax.set_ylabel('[Fe/H]', fontsize=18)
    if i >= len(N_list) - ncols:
        ax.set_xlabel('Age [Gyr]', fontsize=18)

# Removes the empty panels when len(N_list) is not a multiple of ncols
for j in range(len(N_list), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig("AGE-METALLICITY.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# G-dwarf MDF: 0.8 < M < 1.05 Msun still on the MS (largest N)
N = N_list[-1]
M, age, MH, obj_type, M_rem, iterations = load_simulation(N)

sel = (obj_type == 0) & (M > M_G_min) & (M < M_G_max)
MH_G = MH[sel]

bins = bins_FeH     # use the same bins for the observed MDF

# Closed-box prediction: dN/dZ ∝ exp(-Z/y), so dN/d[M/H] ∝ Z exp(-Z/y), normalized to unit area
x_curve = np.linspace(bins[0], 0, 500)
Z_curve = Z_sun * 10**x_curve
MDF_curve = Z_curve * np.exp(-Z_curve / y)
MDF_curve = MDF_curve / integrate(MDF_curve, x_curve)

# Metal-poor fraction ([Fe/H] < -0.5): simulated and analytic closed-box value
f_poor_sim = np.mean(MH_G < metal_poor_threshold)
f_poor_cb = (1 - np.exp(-Z_sun * 10**(metal_poor_threshold) / y)) / (1 - u_f)
print(f'G dwarfs: {len(MH_G)}')
print(f'Fraction [Fe/H] < -0.5: simulated = {f_poor_sim:.3f}, analytic closed box = {f_poor_cb:.3f}')

plt.figure(figsize=(10, 6))
plt.hist(MH_G, bins=bins, density=True, alpha=0.7, color='goldenrod', edgecolor='white',
         label=f'Simulated G dwarfs ({len(MH_G)})')
plt.plot(x_curve, MDF_curve, color='black', label='Closed box (analytic)')

plt.xlabel('[Fe/H]', fontsize=12)
plt.ylabel('Normalized MDF', fontsize=12)
plt.legend(fontsize=10, loc='upper left')

plt.savefig("MDF.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Method comparison: efficiency and runtime
# The efficiency of rejection sampling is the fraction of accepted proposals, N / iterations.
N_common = sorted(set(list_N('rejection')) & set(list_N('inverse')))
print('N with both methods:', N_common)

print(f'{"N":>9} | {"rejection iterations":>20}{"efficiency":>12}{"runtime [s]":>12} | {"inverse iterations":>20}{"runtime [s]":>12}')
for N in N_common:
    it_r = load_simulation(N, 'rejection')[5]
    it_i = load_simulation(N, 'inverse')[5]
    print(f'{N:>9} | {it_r:>20}{N/it_r:>12.5f}{read_runtime(N, "rejection"):>12.2f} | '
          f'{it_i:>20}{read_runtime(N, "inverse"):>12.2f}')

# Theoretical efficiency of rejection sampling: area under the IMF / area of the proposal box
theoretical_efficiency = (I1 + I2) / (IMF(M_min) * (M_max - M_min))
print(f'\nTheoretical efficiency of rejection sampling = {theoretical_efficiency:.5f}')

In [ ]:
# Method comparison: IMF and fractions of each type
N = N_common[-1]
M_r = load_simulation(N, 'rejection')[0]
M_i = load_simulation(N, 'inverse')[0]

m_grid = np.logspace(np.log10(M_min), np.log10(M_max), 2000)
m_grid[0], m_grid[-1] = M_min, M_max      # avoids rounding the first point below M_min
IMF_grid = np.array([IMF(m) for m in m_grid])
IMF_grid = IMF_grid / integrate(IMF_grid, m_grid)
bins = np.logspace(np.log10(M_min), np.log10(M_max), 50)

# Fraction of each type for each method and each N
frac = {met: {k: [] for k in type_names} for met in ['rejection', 'inverse']}
for met in ['rejection', 'inverse']:
    for Nc in N_common:
        obj_type = load_simulation(Nc, met)[3]
        for k in type_names:
            frac[met][k].append(np.sum(obj_type == k) / len(obj_type))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# IMF
ax1.hist(M_r, bins=bins, density=True, histtype='step', linewidth=2, color='crimson', label=f'Rejection sampling')
ax1.hist(M_i, bins=bins, density=True, histtype='step', linewidth=2, color='royalblue', label=f'Inverse transform')
ax1.plot(m_grid, IMF_grid, color='black', label='Kroupa IMF')
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel(r'Mass [$M_\odot$]', fontsize=18)
ax1.set_ylabel(r'$dN/dM$ (normalized)', fontsize=18)
ax1.legend(fontsize=14)

# Fractions as a function of N (zero fractions are hidden, they cannot be drawn on a log scale)
no_zeros = lambda f: np.where(np.array(f) > 0, f, np.nan)
for k in type_names:
    ax2.plot(N_common, no_zeros(frac['rejection'][k]), 'o-', color=type_colors[k], label=f'{type_names[k]} rejection')
    ax2.plot(N_common, no_zeros(frac['inverse'][k]), 's--', color=type_colors[k], mfc='none', label=f'{type_names[k]} inverse')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('N', fontsize=18)
ax2.set_ylabel('Fraction', fontsize=18)
ax2.legend(fontsize=11, ncol=2)

plt.tight_layout()
plt.savefig("SAMPLING_METHODS.png", bbox_inches='tight', dpi=150)
plt.show()

# Difference between methods in Poisson sigmas: for two independent samples of size N,
# sigma(f_r - f_i) = sqrt(n_r + n_i) / N = sqrt(f_r + f_i) / sqrt(N)
print(f'Fractions for N = {N}:')
for k in type_names:
    f_r, f_i = frac['rejection'][k][-1], frac['inverse'][k][-1]
    sigma_diff = np.sqrt(f_r + f_i) / np.sqrt(N)
    print(f'{type_names[k]}: rejection = {f_r:.5f}, inverse = {f_i:.5f}, '
          f'difference = {f_r - f_i:+.5f} ({(f_r - f_i)/sigma_diff:+.1f} sigma)')

In [ ]:
# Reading the observed MDFs (one value of [X/H] per line, generated by the catalogue notebook)
obs_files = {
    'Haywood (2001) [Fe/H]':   ('MDF_obs_Haywood2001_FeH.dat',   'black'),
    'GCS III (2009) [Fe/H]':   ('MDF_obs_GCS3_FeH.dat',          'gray'),
    'Adibekyan (2012) [Fe/H]': ('MDF_obs_Adibekyan2012_FeH.dat', 'royalblue'),
    'Adibekyan (2012) [Mg/H]': ('MDF_obs_Adibekyan2012_MgH.dat', 'crimson'),
}

obs = {}
for name, (filename, color) in obs_files.items():
    if os.path.exists(filename):
        obs[name] = (np.loadtxt(filename, ndmin=1), color)
        print(f'{name}: {len(obs[name][0])} stars')
    else:
        print(f'Warning: missing {filename}')

In [ ]:
# Closed-box MDF vs observations (G-dwarf problem)
N = N_list[-1]
M, age, MH, obj_type, M_rem, iterations = load_simulation(N)
MH_G = MH[(obj_type == 0) & (M > M_G_min) & (M < M_G_max)]

bins = bins_FeH
centers = 0.5 * (bins[1:] + bins[:-1])


def MDF_summary(data):
    """
    Normalized histogram and metrics of an MDF.

    The histogram uses bins_FeH and density=True, so model and observations are compared with
    the same bins and normalized to unit area within them.

    Output: dictionary with the histogram 'h', the bin center of the peak, the median, the
            standard deviation and the metal-poor ([X/H] < metal_poor_threshold) and
            super-solar ([X/H] > 0) fractions.
    """
    h, _ = np.histogram(data, bins=bins, density=True)
    return {'h': h,
            'peak': centers[np.argmax(h)],
            'median': np.median(data),
            'sigma': np.std(data),
            'f_poor': np.mean(data < metal_poor_threshold),
            'f_super': np.mean(data > 0)}


res_model = MDF_summary(MH_G)

# Table of metrics; the last column is the bin with the largest |model - observation| difference
print(f'{"Sample":<28}{"peak":>8}{"median":>9}{"sigma":>8}{"f(<-0.5)":>10}{"f(>0)":>8}   largest discrepancy (model - obs)')
print(f'{"Closed box, N = " + str(N):<28}{res_model["peak"]:>8.2f}{res_model["median"]:>9.2f}{res_model["sigma"]:>8.2f}'
      f'{res_model["f_poor"]:>10.3f}{res_model["f_super"]:>8.3f}')

res_obs = {}
for name, (data, color) in obs.items():
    r = MDF_summary(data)
    res_obs[name] = r
    diff = res_model['h'] - r['h']
    i_max = np.argmax(np.abs(diff))
    print(f'{name:<28}{r["peak"]:>8.2f}{r["median"]:>9.2f}{r["sigma"]:>8.2f}'
          f'{r["f_poor"]:>10.3f}{r["f_super"]:>8.3f}   [X/H] = {centers[i_max]:+.2f} ({diff[i_max]:+.2f})')

# Top: normalized MDFs. Bottom: model - observation per bin (positive = excess in the model)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

# Normalized MDFs
ax1.hist(MH_G, bins=bins, density=True, alpha=0.5, color='goldenrod', edgecolor='white',
         label=f'Closed box, G dwarfs ({len(MH_G)})')
for name, (data, color) in obs.items():
    ax1.hist(data, bins=bins, density=True, histtype='step', linewidth=2, color=color,
             label=f'{name} ({len(data)})')
ax1.axvline(metal_poor_threshold, color='gray', linestyle=':', linewidth=1.5)
ax1.set_ylabel('Normalized MDF', fontsize=12)
ax1.legend(fontsize=10, loc='upper left')

# Model - observation difference
for name, (data, color) in obs.items():
    ax2.stairs(res_model['h'] - res_obs[name]['h'], bins, color=color, linewidth=2)
ax2.axhline(0, color='gray', linewidth=1)
ax2.axvline(metal_poor_threshold, color='gray', linestyle=':', linewidth=1.5)
ax2.set_xlabel('[M/H] model / [X/H] observed', fontsize=12)
ax2.set_ylabel('Model − obs', fontsize=12)

plt.tight_layout()
plt.savefig("MDF_MODEL_VS_OBS.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Chemical evolution with infall: accretion rate, gas fraction and gas metallicity
model_colors = {'Closed box': 'red'}
for name, color in zip(list(models)[1:], plt.cm.viridis(np.linspace(0, 0.8, len(tau_infall_list)))):
    model_colors[name] = color

# Panels: accretion rate, gas fraction and gas metallicity as a function of stellar age
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))

for name, mod in models.items():
    ax1.plot(mod['t'], mod['A'], color=model_colors[name], linewidth=2, label=name)
    ax2.plot(mod['t'], mod['M_gas'] / mod['M_tot'], color=model_colors[name], linewidth=2, label=name)
    with np.errstate(divide='ignore'):
        ax3.plot(T_MW - mod['t'], np.log10(mod['Z'] / Z_sun), color=model_colors[name], linewidth=2, label=name)

ax1.set_xlabel(r'$t$ [Gyr]', fontsize=18)
ax1.set_ylabel(r'$A(t)$ [$M_{\rm tot}(T_{\rm MW})$ Gyr$^{-1}$]', fontsize=16)
ax2.set_xlabel(r'$t$ [Gyr]', fontsize=18)
ax2.set_ylabel(r'$\mu = M_{\rm gas}/M_{\rm tot}$', fontsize=18)
ax3.set_xlabel('Age [Gyr]', fontsize=18)
ax3.set_ylabel(r'[M/H]$_{\rm gas}$ at birth', fontsize=18)
ax3.set_ylim(-3, 0.2)
for ax in (ax1, ax2, ax3):
    ax.tick_params(labelsize=14)
    ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig("INFALL_EVOLUTION.png", bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# G-dwarf MDF: closed box vs infall vs observations
# The same G dwarfs are used for every model; only their birth [M/H] changes.
N = N_list[-1]
M, age, MH, obj_type, M_rem, iterations = load_simulation(N)
sel_G = (obj_type == 0) & (M > M_G_min) & (M < M_G_max)

bins = bins_FeH
centers = 0.5 * (bins[1:] + bins[:-1])

# Birth [M/H] of the G dwarfs in each model (stars born exactly at t = 0 have Z = 0 and are excluded)
MDF_models = {}
for name, mod in models.items():
    MH_mod = birth_MH(mod, age[sel_G])
    MDF_models[name] = MH_mod[np.isfinite(MH_mod)]

print(f'{"Sample":<30}{"peak":>8}{"median":>9}{"sigma":>8}{"f(<-0.5)":>10}{"f(>0)":>8}')
for name, data in list(MDF_models.items()) + [(k, v[0]) for k, v in obs.items()]:
    r = MDF_summary(data)
    print(f'{name.replace("$", "").replace(chr(92) + "tau", "tau"):<30}{r["peak"]:>8.2f}{r["median"]:>9.2f}'
          f'{r["sigma"]:>8.2f}{r["f_poor"]:>10.3f}{r["f_super"]:>8.3f}')

plt.figure(figsize=(10, 7))
for name, data in MDF_models.items():
    plt.hist(data, bins=bins, density=True, histtype='step', linewidth=2.5, color=model_colors[name],
             label=f'{name} (f(<{metal_poor_threshold}) = {np.mean(data < metal_poor_threshold):.2f})')
for name, (data, color) in obs.items():
    plt.hist(data, bins=bins, density=True, histtype='stepfilled', alpha=0.15, color=color,
             label=f'{name} (f(<{metal_poor_threshold}) = {np.mean(data < metal_poor_threshold):.2f})')
plt.axvline(metal_poor_threshold, color='gray', linestyle=':', linewidth=1.5)

plt.xlabel('[M/H] model / [X/H] observed', fontsize=12)
plt.ylabel('Normalized MDF', fontsize=12)
plt.legend(fontsize=10, loc='upper left')

plt.savefig("MDF_INFALL.png", bbox_inches='tight', dpi=150)
plt.show()